# Reference-based selection of a loss--link specification

This notebook implements the publication experiment described in `REFERENCE_SELECTION_PLAN.md`. It fits the full candidate library and uses separate training, diagnostic, and evaluation observations. The evaluation observations do not enter candidate selection.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "src" / "genriesz").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the genriesz repository.")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz.experiments.reference_selection import grids, report, rescaling
from genriesz.experiments.reference_selection.runner import load_experiment, run_experiment

MAX_WORKERS = None
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "experiments" / "results" / "reference_selection"
RUN_DIR = OUTPUT_ROOT / "publication"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save(table: pd.DataFrame, name: str) -> pd.DataFrame:
    if table.empty:
        print(f"{name}: no rows")
        return table
    display(table)
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)
    return table

## 2. E1a: Rescaling a Bregman objective

Replacing a generator $g$ by $\kappa g$ changes the scale of its Bregman criterion. The unpenalized rows show the exact scale identity. The penalized rows report what happens under the numerical penalties used by `GRRGLM`.

The column `alpha_max_deviation` compares the fitted representers with the fit at $\kappa=1$. The column `objective_ratio` reports the corresponding training-objective ratio. The held-out ratio is computed only when the exact inverse link is defined for every held-out observation. If a BP or BKL index leaves its dual domain, the table reports `dual_domain_failure` and leaves the held-out criterion missing. No clipping or substitute generator is used.


In [ ]:
_ = save(rescaling.rescaling_table(), "e1a_rescaling")

## 3. Run the publication experiment

The hidden-direction scales are read from the committed calibration file. The run uses the publication replication counts. Interrupted batches resume from completed output files without changing the seed assigned to any remaining replication. The reporting cells below load results only after every batch listed in the manifest is present.

Every reporting cell reloads the complete run directory. If the manifest or any expected batch is missing, `load_experiment` raises an error before a table or figure is constructed.


In [ ]:
config = grids.experiment_config(max_workers=MAX_WORKERS)
display(pd.DataFrame([scenario.__dict__ for scenario in config.scenarios]))
print(
    f"{len(config.scenarios)} scenarios, "
    f"{sum(config.replications(scenario) for scenario in config.scenarios)} replication jobs"
)

run_experiment(config, RUN_DIR)
tables = load_experiment(RUN_DIR)
{name: frame.shape for name, frame in tables.items()}

## 4. E1b: Comparison of selection rules

Every rule chooses from the same fitted library on the same fold. The comparison is paired, so evaluating another rule does not require another candidate fit.

- `bregman_cv` is the naive rule whose scale dependence is shown in Section 2.
- `lsif_cv` uses a generator-independent squared criterion. It is comparable across candidates, but it measures error in $\alpha$ rather than drift in the target parameter.
- `abs_drift` drops the simultaneous radius and isolates the contribution of $q_a$.
- `fixed_*` are specifications chosen without data-dependent selection.
- `oracle` uses known simulation quantities and serves only as an infeasible benchmark.

Coverage is unconditional: a rule that produces no estimate counts as a non-covering replication. Under the exact ATE branch specification, a BKL candidate is unavailable whenever its dual index leaves the open BKL domain on the training, diagnostic, or evaluation observations. The implementation records that domain failure and does not replace the candidate with bounded BKL.


In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.selection_rule_table(tables), "e1b_selection_rules")

In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.selection_frequency_table(tables), "e1b_selection_frequency")
_ = save(report.failure_table(tables), "numerical_failures")

## 5. E2: is the bias bound valid, and is it tight?

`lower_coverage` is the share of candidates with $|B_a| \le \widehat U_a$; `upper_coverage` checks the other half of Theorem `data_dependent_bias`, $\widehat U_a \le |B_a| + 2(q_a + b_r)$. Reporting only the first cannot distinguish a valid bound from a vacuous one.

`radius_share` and `allowance_share` decompose $\widehat U_a$ and say which term binds. `reference_drift` and `allowance_covers_reference` record whether the theorem's premise $|B_r| \le b_r$ actually held, rather than assuming it.

In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.bias_bound_table(tables), "e2_bias_bounds")

## 6. E3: the oracle inequality

`risk_ratio` is the realized conditional risk of the selected candidate divided by that of the best admissible candidate. Corollary `oracle_remainder` predicts it converges to one when the diagnostic error and the allowance are small relative to the oracle risk.

In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.oracle_regret_table(tables), "e3_oracle_regret")

## 7. E4: Coverage over the calibrated bias family

The hidden-direction scale is calibrated so that a fixed benchmark specification attains a target bias-to-standard-error ratio $t$, which indexes the bounded-normal-mean problem. The benchmark is fixed before selection, so the family does not depend on the selection rule.

The `min` rows report the lowest coverage across the family. The dotted curve gives the theoretical coverage of the ordinary Wald interval, $2\Phi(1.96 - t) - 1$. The other rows report the bias-aware single-split interval and the conservative cross-fitted interval.

In [ ]:
tables = load_experiment(RUN_DIR)
uniform = report.uniform_coverage_table(tables)
_ = save(uniform, "e4_uniform_coverage")

# The headline row: the least favourable point of the family, with the standard
# error and interval length taken from that same scenario rather than from a
# column-wise minimum that would mix scenarios.
_ = save(report.worst_case_coverage_table(tables), "e4_worst_case_coverage")

In [ ]:
tables = load_experiment(RUN_DIR)
uniform = report.uniform_coverage_table(tables)
sweep = uniform.copy()
if not sweep.empty:
    for sample_size, frame in sweep.groupby("sample_size"):
        frame = frame.sort_values("target_t")
        figure, axis = plt.subplots(figsize=(7.0, 4.5))
        for label, column in (
            ("Ordinary Wald", "wald_split_coverage"),
            ("Bias-aware (single split)", "bias_aware_split_coverage"),
            ("Conservative cross-fitted", "conservative_cf_coverage"),
        ):
            if column not in frame:
                continue
            error = frame.get(f"{column}_mcse")
            axis.errorbar(
                frame["target_t"],
                frame[column],
                yerr=None if error is None else 1.96 * error,
                marker="o",
                capsize=3,
                label=label,
            )
        grid = np.linspace(0.0, float(frame["target_t"].max()), 100)
        axis.plot(
            grid,
            stats.norm.cdf(1.959964 - grid) - stats.norm.cdf(-1.959964 - grid),
            linestyle=":",
            color="grey",
            label="Wald (theoretical)",
        )
        axis.axhline(0.95, linestyle="--", linewidth=1.0, color="black")
        axis.set_xlabel("Calibrated bias-to-standard-error ratio $t$")
        axis.set_ylabel("Coverage probability")
        axis.set_ylim(0.0, 1.02)
        axis.set_title(f"Coverage across the bias family (n={sample_size})")
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURE_DIR / f"e4_coverage_n{sample_size}.pdf", bbox_inches="tight")
        plt.show()

## 8. E5: how much rests on the reference allowance?

Two things are varied: the reference (`truth`, `correct`, `misspecified`, and `min`, the minimum bound over the two estimated ones) and the scale $\rho$ applied to the honest allowance. Setting $\rho = 0$ removes the allowance entirely and shows how much of the guarantee it carries.

The `misspecified` reference keeps the *same allowance formula* while dropping terms from both of its nuisance models, so its stated $b_r$ no longer bounds its drift. The `correct` reference includes the hidden direction in both nuisances and stays valid as $t$ grows; without that, every candidate bound would fail at once and the sweep would measure nothing.

The minimum bound in Proposition `several_references` is valid only when every included reference satisfies its allowance. The pairwise check reports how often the estimated references conflict with their stated allowances.

In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.reference_robustness_table(tables), "e5_reference_robustness")

In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.reference_check_table(tables), "e5_reference_check")

## 9. E6: Length of bias-aware intervals

The column `bound_to_se` is $\widehat U_{\widehat a} / \widehat{\mathrm{se}}_{\widehat a}$, and `bias_aware_over_wald` is the ratio of interval lengths. Corollary `oa:bias_aware_length` implies that the length ratio approaches one when the standardized bound approaches zero.

In [ ]:
tables = load_experiment(RUN_DIR)
_ = save(report.interval_length_table(tables), "e6_interval_length")

## 10. Output files

Parquet results are written under `notebooks/experiments/results/reference_selection/publication/`. A table or figure is constructed only after all manifest batches are present. The notebook also writes the displayed CSV tables and PDF copies of the figures. Figure layout and display code remain in this notebook.

The reported intervals are `conservative_cf`, which is the cross-fitted interval supported by the manuscript, and `bias_aware_split`, which is the single-split interval covered by the corresponding theorem. The diagnostic field `bias_aware_pooled` is not used as a theoretically supported confidence interval.
